# Projet Python pour la data science (2026)

# Alix HERITIER, Ivan TISSOT

In [242]:
!pip install -r requirements.txt -q # Installation des librairies nécessaires
!pip install openpyxl

# Importation des librairies
import pandas as pd 
import os as os
import openpyxl as openpyxl
import geopandas as gpd
from cartiflette import carti_download 

# Importation des fonctions
from extraire_url_zip import extract_zip_from_url
from d_chomage import donnees_chomage
from d_cs import donnees_cs
from carte_ecarts_dep import carte_ecarts_departement
#from d_mediane import donnees_mediane

# Dossiers pour extraire les fichiers
extract_path = '/home/onyxia/ENSAI-2A-projet-sport/ttt/'
folder_path = '/home/onyxia/ENSAI-2A-projet-sport/ttt/'

# Importation des données électorales

In [ ]:
de = pd.read_csv('https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb')

In [ ]:
# Vérification de la présence de données manquantes
print("Valeurs manquantes par colonne") 
print(de.isnull().sum())

# On veut voir quels sont les noms associés aux prénoms manquants
# Garder uniquement les lignes où le prénom est manquant
de_prenom_manquant = de[de['prenom'].isnull()]

# Affichage des noms associés aux prénoms manquants
print(f"Nombre de lignes avec prénom manquant : {len(de_prenom_manquant)}")
print("\nNoms associés aux prénoms manquants ")
print(de_prenom_manquant['nom'].value_counts())

# On constate que les noms associés aux prénoms manquants sont : "abstentions" ; "blancs" ; "nuls"
# Nous enlèverons ces données plus tard donc ces données manquantes ne sont pas problématiques pour le devoir maison ! 

In [ ]:
# On met les codes en texte avec le bon nombre de chiffres
de["code_departement"] = de["code_departement"].astype(str).str.zfill(2)

# On crée la colonne candidat = prénom + nom
de["prenom"] = de["prenom"].fillna("").astype(str).str.strip()
de["nom"] = de["nom"].fillna("").astype(str).str.strip()

de["candidat"] = de["prenom"] + " " + de["nom"]
de["candidat"] = de["candidat"].str.strip()


In [ ]:
# On ne prend que les candidats ayant un prénom et un nom 
non_candidats = ["abstentions", "blancs", "nuls"]
de_candidats = de[de["candidat"].isin(non_candidats) == False]


In [ ]:
# Calcul des votes par département et par candidat
scores_departementaux = (
    de_candidats.groupby(["code_departement", "candidat"], as_index=False)["voix"]
    .sum()
    .rename(columns={"voix": "votes_departemental"})
)

# Pivot : mettre les candidats en colonnes et les départements en lignes
scores_departementaux_pivot = scores_departementaux.pivot(index="code_departement", columns="candidat", values="votes_departemental")
# Enlever le nom 'candidat' des colonnes
scores_departementaux_pivot.columns.name = None

# Réinitialiser l'index pour remettre 'code_departement' comme colonne
scores_departementaux_pivot = scores_departementaux_pivot.reset_index()

scores_departementaux_pivot['somme'] = scores_departementaux_pivot.iloc[:, 1:13].sum(axis=1)
scores_departementaux_pivot = scores_departementaux_pivot.iloc[0:96, 0:14]
for col in scores_departementaux_pivot.columns:
    if col != 'somme' and col != 'code_departement':
        scores_departementaux_pivot[f's_{col}'] = scores_departementaux_pivot[col] / scores_departementaux_pivot['somme']
print(len(scores_departementaux_pivot.columns))

In [ ]:
# Création des colonnes scores
dele = scores_departementaux_pivot
dele = dele.iloc[0:96, [0,14,15,16,17,18,19,20,21,22,23,24,25]]
new_column_names = ['code', 'HIDALGO', 'MACRON', 'ROUSSEL', 'LASSALLE', 'MELENCHON', 'LE PEN', 
                    'ARTHAUD', 'DUPONT-AIGNAN', 'POUTOU', 'PECRESSE', 'JADOT', 'ZEMMOUR']
dele.columns = new_column_names

# Importation des données sur le chômage

In [ ]:
url = 'https://www.insee.fr/fr/statistiques/fichier/2134411/TCRD_087.xlsx'
dc = donnees_chomage(url)

In [ ]:
Importation des données:
- médiane du niveau vie (€)
- taux de pauvreté-Ensemble (%)
- rapport interdécile 9ᵉ décile/1ᵉʳ décile
- part des revenus dactivité
- part des ménages fiscaux imposés (%)

In [ ]:
# URL du fichier ZIP
url = 'https://www.insee.fr/fr/statistiques/fichier/6692392/base-cc-filosofi-2020_XLSX.zip'

# Appel de la fonction
extract_zip_from_url(url, extract_path)

# Liste des fichiers chargés
for f in os.listdir(folder_path):
    print(f)

In [ ]:
# Charger le fichier Excel en utilisant le chemin complet
dme = pd.read_excel(os.path.join(folder_path, 'base-cc-filosofi-2020-geo2023.xlsx'),sheet_name='DEP',decimal=',')
dme = dme.iloc[5:101, [0,4,5,6,15,28]]
dme = dme.rename(columns={dme.columns[0]: 'code', 
                          dme.columns[1]: 'Mediane_nivvie', 
                          dme.columns[2]: 'Part_menages_fiscaux_imposes',
                          dme.columns[3]: 'Taux_pauvrete',
                          dme.columns[4]: 'Part_des_revenus_dactivite',
                          dme.columns[5]: 'Rapport_interdecile'})


# Importation des niveaux de diplomes

In [ ]:
# URL du fichier ZIP
url = 'https://www.insee.fr/fr/statistiques/fichier/1893149/pop-16ans-dipl6822.zip'

# Appel de la fonction
extract_zip_from_url(url, extract_path)

# Liste des fichiers chargés
for f in os.listdir(folder_path):
    print(f)

In [ ]:
# Charger le fichier Excel en utilisant le chemin complet
dip = pd.read_excel(os.path.join(folder_path, 'pop-16ans-dipl6822.xlsx'), sheet_name='DEP_2022')
dip = dip.iloc[14:110, 1:32]

In [ ]:
dip['rapport_sommes'] = dip.iloc[:, 2:18].sum(axis=1) / dip.iloc[:, 2:30].sum(axis=1)
dip = dip.iloc[0:96, [0,30]]
dip = dip.rename(columns={dip.columns[0]: 'code',
                          dip.columns[1]: 'Part_inf_bac'})

# Importation des données sur les CS

In [ ]:
url = 'https://www.insee.fr/fr/statistiques/fichier/2012721/TCRD_014.xlsx'
dcs = donnees_cs(url)

# Appariement des 5 tables sur les départements

In [239]:
# Fusionner dcs avec dip sur la colonne 'code'
base = pd.merge(dcs, dip, on='code', how='left')
base = pd.merge(base, dc, on='code', how='left')
base = pd.merge(base, dme, on='code', how='left')
base = pd.merge(base, dele, on='code', how='left')

# Afficher le résultat final
# 1+6 (cs) +1 (dip) +1 (dc)+5 (dme)+12 (dele)=26 variables
# 95 observations
base

,code,cs_agri,cs_arti,cs_cadr,cs_inte,cs_empl,cs_ouvr,Part_inf_bac,Taux_chomage_2024,Mediane_nivvie,...,ROUSSEL,LASSALLE,MELENCHON,LE PEN,ARTHAUD,DUPONT-AIGNAN,POUTOU,PECRESSE,JADOT,ZEMMOUR
0,01,0.9,6.5,16.3,27.1,25.3,23.2,0.487387,5.6,24030,...,0.017831,0.032658,0.173657,0.260507,0.004979,0.027019,0.006522,0.052765,0.047573,0.082667
1,02,1.7,4.9,8.7,22.8,29.3,30.3,0.612701,10.5,20300,...,0.022448,0.024329,0.154863,0.392470,0.007666,0.021778,0.007967,0.041074,0.026608,0.068705
2,03,3,6.5,9.6,22.9,30.5,26.2,0.584148,7.9,20990,...,0.043661,0.041849,0.166777,0.270576,0.007308,0.022672,0.008083,0.055492,0.032169,0.066473
3,04,2.7,10.1,12.4,25.8,27.8,20.3,0.499203,7.9,21130,...,0.028144,0.044568,0.226058,0.269024,0.005223,0.025899,0.008947,0.039655,0.040928,0.081979
4,05,2.7,9.6,10.7,28.2,29.9,18.3,0.461219,6.2,21420,...,0.022323,0.044889,0.228654,0.228399,0.004963,0.024839,0.009289,0.052311,0.058132,0.071479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,91,0.1,4.6,24,27.4,26.4,16,0.407806,6.5,24410,...,0.022658,0.019515,0.281221,0.177937,0.004758,0.025554,0.007238,0.055545,0.049689,0.066030
92,92,0,4.6,44,22.6,19.9,7.5,0.266366,6,28810,...,0.016999,0.014534,0.257686,0.083656,0.002976,0.012576,0.004781,0.080325,0.060798,0.081009
93,93,0,5,17.9,23.1,30.8,20.4,0.484342,10.3,18470,...,0.021434,0.012529,0.490898,0.118831,0.005074,0.011599,0.006744,0.032184,0.035629,0.051494
94,94,0,4.9,28.6,25.3,26.3,13.1,0.372042,7.3,23540,...,0.025386,0.015184,0.326669,0.118153,0.004251,0.015888,0.006332,0.055210,0.054161,0.073728


# Régressions linéaires

# MACRON

In [ ]:
base_macron = base.iloc[0:96, [1,2,3,4,5,6,7,8,9,10,11,12,13,15]]

In [ ]:
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# Sélectionner les variables indépendantes (les 14 premières colonnes)
X = base_macron.iloc[:, :-1]  # Toutes les colonnes sauf la dernière (15ème variable)

# Sélectionner la variable dépendante (la 15ème variable)
y = base_macron.iloc[:, -1]  # La dernière colonne (15ème variable)

# Centrer et réduire les variables indépendantes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Ajouter une constante (biais) aux variables indépendantes (intercept)
X_scaled = sm.add_constant(X_scaled)

# Créer le modèle de régression linéaire
model = sm.OLS(y, X_scaled)  # OLS = Ordinary Least Squares (moindres carrés ordinaires)

# Ajuster le modèle
results = model.fit()

# Afficher les résultats de la régression
print(results.summary())

In [243]:
import matplotlib.pyplot as plt

departement_borders = carti_download(
    values=["France"],
    crs=4326,
    borders="DEPARTEMENT",
    vectorfile_format="geojson",
    simplification=50,
    filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
    source="EXPRESS-COG-CARTO-TERRITOIRE",
    year=2022)


carte_ecarts_departement(base, departement_borders, "MACRON")


NameError: name 'base' is not defined

# LE PEN

In [ ]:
base_le_pen = base.iloc[0:96, [1,2,3,4,5,6,7,8,9,10,11,12,13,19]]

In [ ]:
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# Sélectionner les variables indépendantes (les 14 premières colonnes)
X = base_le_pen.iloc[:, :-1]  # Toutes les colonnes sauf la dernière (15ème variable)

# Sélectionner la variable dépendante (la 15ème variable)
y = base_le_pen.iloc[:, -1]  # La dernière colonne (15ème variable)

# Centrer et réduire les variables indépendantes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Ajouter une constante (biais) aux variables indépendantes (intercept)
X_scaled = sm.add_constant(X_scaled)

# Créer le modèle de régression linéaire
model = sm.OLS(y, X_scaled)  # OLS = Ordinary Least Squares (moindres carrés ordinaires)

# Ajuster le modèle
results = model.fit()

# Afficher les résultats de la régression
print(results.summary())

# MELENCHON

In [ ]:
base_melenchon = base.iloc[0:96, [1,2,3,4,5,6,7,8,9,10,11,12,13,18]]

In [ ]:
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# Sélectionner les variables indépendantes (les 14 premières colonnes)
X = base_melenchon.iloc[:, :-1]  # Toutes les colonnes sauf la dernière (15ème variable)

# Sélectionner la variable dépendante (la 15ème variable)
y = base_melenchon.iloc[:, -1]  # La dernière colonne (15ème variable)

# Centrer et réduire les variables indépendantes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Ajouter une constante (biais) aux variables indépendantes (intercept)
X_scaled = sm.add_constant(X_scaled)

# Créer le modèle de régression linéaire
model = sm.OLS(y, X_scaled)  # OLS = Ordinary Least Squares (moindres carrés ordinaires)

# Ajuster le modèle
results = model.fit()

# Afficher les résultats de la régression
print(results.summary())